# Blocks

> Sentence-sized change blocks with optional LLM comments.

In [ ]:
#| default_exp blocks

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import re
import json
import httpx
from typing import List, Dict, Any, Optional

from text_diff.diff import compute_diff
from text_diff.clean import clean_newlines

## Why blocks

A word-level diff is precise but hard to review: one rephrased sentence can become
five or six scattered fragments. A *block* is the unit a person actually reviews:
every change inside one sentence (or clause) is gathered into a single block, which
keeps the word-level ops for highlighting but is accepted or rejected as a whole.

Each block is a dict:

- `id`: stable string id
- `type`: `"equal"` (unchanged text) or `"change"`
- `old_text` / `new_text`: the full text of the block before and after
- `ops`: the word-level ops inside the block, for inline highlighting
- `comment`: a short reason for the change (empty until `add_comments` runs)
- `status`: `"pending"` for a change nobody has reviewed yet, then `"accepted"` or `"rejected"`;
  equal blocks are always `"accepted"`

## Splitting at boundaries

Blocks may only be cut inside *unchanged* text. We split equal ops at sentence
boundaries (end punctuation followed by whitespace, or any newline). With
`granularity="clause"` we also split after commas, semicolons and colons.

In [ ]:
#| export
class _Shifted:
    "A regex match with positions shifted back by `n`."
    def __init__(self, m, n): self._m, self._n = m, n
    def start(self): return self._m.start() - self._n
    def end(self): return self._m.end() - self._n
    def group(self): return self._m.group()

_BOUNDARY = {
    "sentence": re.compile(r'(?<=[.!?])[ \t]+|\n+'),
    "clause": re.compile(r'(?<=[.!?,;:])[ \t]+|\n+'),
}

def _split_equal(text: str, granularity: str, prev: str = "") -> List[Dict[str, Any]]:
    """Split unchanged text into pieces, marking the separators that end a unit.

    `prev` is the character just before `text` (it may sit inside a change, as in
    "mat." -> "rug."), so a boundary right at the start of `text` is still found."""
    pattern = _BOUNDARY[granularity]
    pieces, pos = [], 0
    for m in pattern.finditer(prev + text, len(prev)):
        m = _Shifted(m, len(prev))
        if m.start() > pos:
            pieces.append({"text": text[pos:m.start()], "boundary": False})
        pieces.append({"text": m.group(), "boundary": True})
        pos = m.end()
    if pos < len(text):
        pieces.append({"text": text[pos:], "boundary": False})
    return pieces

In [ ]:
assert _split_equal("One. Two", "sentence") == [
    {"text": "One.", "boundary": False},
    {"text": " ", "boundary": True},
    {"text": "Two", "boundary": False},
]
assert _split_equal("a, b", "sentence") == [{"text": "a, b", "boundary": False}]
assert [p["text"] for p in _split_equal("a, b", "clause")] == ["a,", " ", "b"]
assert _split_equal("x\n\ny", "sentence")[1] == {"text": "\n\n", "boundary": True}
# punctuation inside the preceding change still ends the sentence
assert _split_equal(" It was", "sentence", prev=".")[0] == {"text": " ", "boundary": True}

## Building blocks

In [ ]:
#| export
def _op_old(op): return op.get("text", "") if op["type"] in ("equal", "remove") else op.get("old_text", "")
def _op_new(op): return op.get("text", "") if op["type"] in ("equal", "add") else op.get("new_text", "")

def _merge_ops(ops: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    "Join consecutive equal ops."
    out = []
    for op in ops:
        if out and op["type"] == "equal" and out[-1]["type"] == "equal":
            out[-1] = {"type": "equal", "text": out[-1]["text"] + op["text"]}
        else:
            out.append(dict(op))
    return out

def ops_to_blocks(ops: List[Dict[str, Any]], granularity: str = "sentence") -> List[Dict[str, Any]]:
    """Gather word-level diff ops into reviewable blocks.

    All changes within one sentence (or clause) become a single `change` block;
    text between them is kept inside the block so it reads as a whole sentence."""
    if granularity not in _BOUNDARY:
        raise ValueError(f"granularity must be one of {list(_BOUNDARY)}")

    units, current, prev = [], [], ""
    for op in ops:
        if op["type"] != "equal":
            current.append({k: v for k, v in op.items() if k not in ("accepted", "status")})
            prev = (_op_new(op) or _op_old(op))[-1:] or prev
            continue
        text = op.get("text", "")
        pieces = _split_equal(text, granularity, prev)
        prev = text[-1:] or prev
        for piece in pieces:
            if piece["boundary"]:
                if current: units.append(current)
                units.append([{"type": "equal", "text": piece["text"]}])
                current = []
            else:
                current.append({"type": "equal", "text": piece["text"]})
    if current: units.append(current)

    blocks = []
    for unit in units:
        unit = _merge_ops(unit)
        old = "".join(_op_old(o) for o in unit)
        new = "".join(_op_new(o) for o in unit)
        changed = any(o["type"] != "equal" for o in unit)
        if not changed and blocks and blocks[-1]["type"] == "equal":
            blocks[-1]["old_text"] += old
            blocks[-1]["new_text"] += new
            blocks[-1]["ops"] = _merge_ops(blocks[-1]["ops"] + unit)
            continue
        blocks.append({
            "type": "change" if changed else "equal",
            "old_text": old,
            "new_text": new,
            "ops": unit,
            "comment": "",
            "status": "pending" if changed else "accepted",
        })
    for i, b in enumerate(blocks):
        b["id"] = str(i)
    return blocks

def compute_blocks(original: str, rewritten: str, granularity: str = "sentence") -> List[Dict[str, Any]]:
    "Diff two texts and return reviewable blocks (no LLM calls)."
    return ops_to_blocks(clean_newlines(compute_diff(original, rewritten)), granularity)

In [ ]:
original = "The cat sat on the mat. It was very very happy. The end."
rewritten = "The cat lay on the rug. It was very happy. The end."
blocks = compute_blocks(original, rewritten)
changes = [b for b in blocks if b["type"] == "change"]
# two edits in the first sentence -> one block; one edit in the second -> one block
assert len(changes) == 2
assert changes[0]["old_text"] == "The cat sat on the mat."
assert changes[0]["new_text"] == "The cat lay on the rug."
assert changes[1]["old_text"] == "It was very very happy."
# blocks rebuild both texts exactly
assert "".join(b["old_text"] for b in blocks) == original
assert "".join(b["new_text"] for b in blocks) == rewritten
assert all(b["status"] == "accepted" for b in blocks if b["type"] == "equal")
assert all(b["status"] == "pending" for b in changes)
assert [b["id"] for b in blocks] == [str(i) for i in range(len(blocks))]
blocks

In [ ]:
# unchanged text collapses into a single equal block
assert compute_blocks("Same. Text.", "Same. Text.") == [
    {"type": "equal", "old_text": "Same. Text.", "new_text": "Same. Text.",
     "ops": [{"type": "equal", "text": "Same. Text."}], "comment": "", "status": "accepted", "id": "0"}
]
# a change that merges two sentences becomes one block spanning both
b = compute_blocks("Short one. Short two.", "Short one and short two.")
assert [x["type"] for x in b] == ["change"]
# clause granularity gives smaller blocks
o, r = "First part, second part.", "First bit, second bit."
assert len([x for x in compute_blocks(o, r) if x["type"] == "change"]) == 1
assert len([x for x in compute_blocks(o, r, "clause") if x["type"] == "change"]) == 2
# paragraphs are hard boundaries
b = compute_blocks("A b\n\nC d", "A x\n\nC y")
assert len([x for x in b if x["type"] == "change"]) == 2
# rebuild property on a longer text
o = "Line one is here.\nLine two, with a clause. And more!\n\nNew para? Yes."
r = "Line one is there.\nLine two, with clauses. And more!\n\nA new para? Yes indeed."
for g in ("sentence", "clause"):
    bl = compute_blocks(o, r, g)
    assert "".join(x["old_text"] for x in bl) == o
    assert "".join(x["new_text"] for x in bl) == r

## Applying decisions

Blocks are plain dicts, so a UI can change them directly, but these helpers keep
them consistent. Each returns a new list and leaves the input untouched.

- `set_status`: mark a change `"pending"`, `"accepted"` or `"rejected"`
- `edit_block`: let the user tweak a suggestion before accepting it; the word-level
  `ops` are recomputed so highlighting still matches
- `blocks_to_text`: the text with every accepted change applied and everything else
  (pending or rejected) left as the original

In [ ]:
#| export
def blocks_to_text(blocks: List[Dict[str, Any]]) -> str:
    "Original text with accepted change blocks swapped for their new text."
    return "".join(b["new_text"] if b["status"] == "accepted" else b["old_text"] for b in blocks)

STATUSES = ("pending", "accepted", "rejected")

def _find(blocks, block_id):
    for i, b in enumerate(blocks):
        if b["id"] == block_id: return i
    raise KeyError(f"No block with id {block_id!r}")

def set_status(blocks: List[Dict[str, Any]], block_id: str, status: str) -> List[Dict[str, Any]]:
    "Return blocks with one change block set to `status`."
    if status not in STATUSES: raise ValueError(f"status must be one of {STATUSES}")
    i = _find(blocks, block_id)
    if blocks[i]["type"] != "change": raise ValueError("Only change blocks have a status to set")
    return blocks[:i] + [{**blocks[i], "status": status}] + blocks[i+1:]

def edit_block(blocks: List[Dict[str, Any]], block_id: str, new_text: str) -> List[Dict[str, Any]]:
    "Return blocks with one change block's suggestion replaced by `new_text`."
    i = _find(blocks, block_id)
    b = blocks[i]
    if b["type"] != "change": raise ValueError("Only change blocks can be edited")
    ops = clean_newlines(compute_diff(b["old_text"], new_text))
    return blocks[:i] + [{**b, "new_text": new_text, "ops": _merge_ops(ops)}] + blocks[i+1:]

In [ ]:
bl = compute_blocks(original, rewritten)
assert blocks_to_text(bl) == original
first, second = [b["id"] for b in bl if b["type"] == "change"]
bl = set_status(bl, first, "accepted")
assert blocks_to_text(bl).startswith("The cat lay on the rug. It was very very happy.")
# rejected and pending both keep the original
bl = set_status(bl, second, "rejected")
assert blocks_to_text(bl) == "The cat lay on the rug. It was very very happy. The end."
# helpers don't mutate their input
fresh = compute_blocks(original, rewritten)
set_status(fresh, first, "accepted")
assert fresh[int(first)]["status"] == "pending"
# edit a suggestion, then accept it
bl = edit_block(fresh, second, "It was so happy.")
assert bl[int(second)]["new_text"] == "It was so happy."
assert {"type": "replace", "old_text": "very very", "new_text": "so"} in bl[int(second)]["ops"]
bl = set_status(bl, second, "accepted")
assert blocks_to_text(bl) == "The cat sat on the mat. It was so happy. The end."
# bad input is rejected loudly
for call in (lambda: set_status(bl, first, "maybe"), lambda: set_status(bl, "999", "accepted"),
             lambda: edit_block(bl, str(len(bl) - 1), "x")):
    try: call(); raise AssertionError("should have raised")
    except (ValueError, KeyError): pass

## Comments

`add_comments` asks the LLM for one short reason per change block, in a single call.
It is optional and never fatal: if the call fails, blocks come back without comments.

In [ ]:
#| export
def comment_prompt(blocks: List[Dict[str, Any]], goal: str = "") -> str:
    "Build the user prompt listing each change block."
    items = [{"id": b["id"], "before": b["old_text"], "after": b["new_text"]}
             for b in blocks if b["type"] == "change"]
    return (
        (f"The writer's goal: {goal}\n\n" if goal else "")
        + "For each edit below, give a short reason (under 12 words) for the change, "
        "written for the writer, e.g. \"Removes repetition\" or \"More formal tone\".\n\n"
        + json.dumps(items, indent=2, ensure_ascii=False)
        + '\n\nReturn ONLY a JSON object mapping each id to its reason, e.g. {"1": "..."}.'
    )

def parse_comments(content: str) -> Dict[str, str]:
    "Pull the JSON object out of an LLM reply, tolerating code fences and stray text."
    start, end = content.find("{"), content.rfind("}") + 1
    if start == -1 or end == 0: return {}
    data = json.loads(content[start:end])
    return {str(k): str(v).strip() for k, v in data.items()}

def apply_comments(blocks: List[Dict[str, Any]], comments: Dict[str, str]) -> List[Dict[str, Any]]:
    "Return blocks with `comment` filled in for change blocks that have one."
    return [{**b, "comment": comments.get(b["id"], b["comment"])} if b["type"] == "change" else b
            for b in blocks]

In [ ]:
assert parse_comments('```json\n{"1": "Tighter wording", "3": " Fixes typo "}\n```') == {"1": "Tighter wording", "3": "Fixes typo"}
assert parse_comments("no json here") == {}
bl = compute_blocks(original, rewritten)
ids = [b["id"] for b in bl if b["type"] == "change"]
out = apply_comments(bl, {ids[0]: "Clearer verb", "999": "ignored"})
assert out[int(ids[0])]["comment"] == "Clearer verb"
assert all(b["comment"] == "" for b in out if b["id"] != ids[0])
assert '"before": "The cat sat on the mat."' in comment_prompt(bl, "be concise")

In [ ]:
#| export
async def add_comments(
    blocks: List[Dict[str, Any]],
    api_key: str,
    goal: str = "",
    model: str = "",
) -> List[Dict[str, Any]]:
    "Ask the LLM for a one-line reason per change block. Returns blocks unchanged on failure."
    if not any(b["type"] == "change" for b in blocks): return blocks
    from text_diff.llm import DEFAULT_MODEL
    payload = {
        "model": model or DEFAULT_MODEL,
        "messages": [
            {"role": "system", "content": "You explain edits to a writer. Reply with JSON only."},
            {"role": "user", "content": comment_prompt(blocks, goal)},
        ],
        "temperature": 0.2,
    }
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    try:
        async with httpx.AsyncClient() as client:
            r = await client.post("https://openrouter.ai/api/v1/chat/completions",
                                  json=payload, headers=headers, timeout=60.0)
            r.raise_for_status()
            content = r.json()["choices"][0]["message"]["content"]
        return apply_comments(blocks, parse_comments(content))
    except Exception as e:
        print(f"Commenting failed: {e}")
        return blocks

## End to end

In [ ]:
#| export
async def improve_text(
    text: str,
    goal: str = "",
    api_key: str = "",
    model: str = "",
    comments: bool = True,
    granularity: str = "sentence",
) -> List[Dict[str, Any]]:
    """Rewrite `text` towards `goal` and return reviewable blocks.

    One LLM call for the rewrite, plus one for comments when `comments=True`."""
    import os
    from text_diff.llm import get_llm_rewrite
    api_key = api_key or os.environ.get("OPENROUTER_API_KEY", "")
    if not api_key: raise ValueError("No API key: pass api_key or set OPENROUTER_API_KEY.")
    rewritten = await get_llm_rewrite(goal or "Improve clarity and flow.", text, api_key, model)
    blocks = compute_blocks(text, rewritten, granularity)
    if comments: blocks = await add_comments(blocks, api_key, goal, model)
    return blocks

```python
blocks = await improve_text("The cat sat on the mat and looked around the room slowly.",
                            goal="make it more concise")
for b in blocks:
    if b["type"] == "change":
        print(b["old_text"], "->", b["new_text"], "|", b["comment"])
```

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()